# LangChain — Orchestrating LLM Applications with Chains, Agents & RAG

---

## What Is This Notebook About?

**LangChain** is a framework for building applications powered by large language models (LLMs). While the OpenAI SDK lets you call one model once, LangChain lets you chain multiple calls, inject data, use agents that think and act, and build complete applications.

Think of the OpenAI SDK as a single instrument; LangChain is the entire orchestra — conductors, sheet music, and instruments all coordinated to produce something bigger than any one call could.

By the end of this notebook you will understand:
- LangChain's core abstractions: LLMs, Prompts, Chains, Memory
- LCEL (LangChain Expression Language) — composing pipelines with `|`
- RAG (Retrieval Augmented Generation) from scratch
- LangChain Agents — LLMs that plan and execute multi-step tasks
- Memory systems for conversational applications
- A mini-project: Document Q&A system

---

## Real-World Analogy: Assembly Line vs Single Worker

Making a car requires many workers:
- Worker 1: Weld the frame
- Worker 2: Install engine
- Worker 3: Add interior
- Inspector: Check quality at each stage

**OpenAI SDK** = one skilled worker. **LangChain** = the assembly line that coordinates all workers:
- Chain 1: Understand user query
- Chain 2: Search documents for relevant info
- Chain 3: Generate answer with context
- Chain 4: Check if answer is grounded in sources

Each step feeds into the next, and the whole pipeline can be configured, swapped, and extended without rewriting.

---

## Prerequisites
- Python basics
- OpenAI SDK notebook (API keys, chat completions, embeddings)

---

## Table of Contents
1. Installation & Setup
2. Core Concepts Overview
3. LCEL — The Pipe Operator Pipeline
4. Prompt Templates
5. Output Parsers
6. Memory Systems
7. RAG — Retrieval Augmented Generation
8. Agents — LLMs That Plan and Act
9. LangChain vs LlamaIndex
10. Common Pitfalls
11. Mini Project: Document Q&A System
12. Interview Q&A
13. Resources

---

## Official Resources
- **Docs**: https://python.langchain.com/
- **GitHub**: https://github.com/langchain-ai/langchain
- **YouTube (LangChain Crash Course)**: https://www.youtube.com/watch?v=aywZrzNaKjs
- **LangSmith (debugging)**: https://smith.langchain.com/
- **LCEL Guide**: https://python.langchain.com/docs/expression_language/

## 1. Installation & Setup

In [ ]:
# Install:
# pip install langchain langchain-openai langchain-community
# pip install faiss-cpu     # Vector store for RAG
# pip install tiktoken      # Token counting

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

try:
    import langchain
    from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
    from langchain_core.runnables import RunnablePassthrough, RunnableLambda
    LC_AVAILABLE = True
    print(f"LangChain version: {langchain.__version__}")
except ImportError:
    LC_AVAILABLE = False
    print("LangChain not installed. Run: pip install langchain langchain-openai")
    print("All cells simulate output for learning purposes.")

try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    import openai
    OPENAI_AVAILABLE = True
    API_KEY = os.getenv('OPENAI_API_KEY', '')
    HAS_KEY = bool(API_KEY)
except ImportError:
    OPENAI_AVAILABLE = False
    HAS_KEY = False

CAN_CALL = LC_AVAILABLE and OPENAI_AVAILABLE and HAS_KEY

print(f"\nLangChain: {'✓' if LC_AVAILABLE else '✗'}  OpenAI: {'✓' if OPENAI_AVAILABLE else '✗'}  API Key: {'✓' if HAS_KEY else '✗'}")

if CAN_CALL:
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.7)
    embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
    print("LLM and embeddings ready!")
else:
    llm = None
    embeddings_model = None
    print("Simulating all LangChain outputs.")

## 2. Core Concepts Overview

LangChain is built around these key abstractions:

| Abstraction | Description | Analogy |
|-------------|-------------|--------|
| **LLM / ChatModel** | The AI brain | The worker |
| **PromptTemplate** | Reusable prompt with variables | A form with blanks to fill in |
| **Chain** | Sequence of steps | Assembly line |
| **OutputParser** | Extract structured data from LLM output | Quality inspector |
| **Memory** | Conversation history storage | Short-term memory |
| **Retriever** | Fetch relevant documents | Librarian |
| **Agent** | LLM that decides what to do next | Manager who delegates tasks |
| **Tool** | Function the agent can call | Worker with a specialization |
| **Vector Store** | Database of embeddings | Filing cabinet organized by meaning |

### LCEL (LangChain Expression Language)

LCEL is the glue that connects all these pieces using the `|` (pipe) operator:

```python
chain = prompt | llm | output_parser
result = chain.invoke({"input": "Hello"})
```

This reads left-to-right: prompt template formats the input → sends to LLM → parser extracts structured output. It's like Unix pipes (`cat file | grep error | sort`).

In [ ]:
# ── LCEL: Building Chains with the Pipe Operator ──────────────────────

# ── Core Concept Visualization ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

components = [
    ('Input\n{topic}', '#3498db', 'User input\nor variables'),
    ('Prompt\nTemplate', '#9b59b6', 'Formats input\ninto full prompt'),
    ('LLM\n(ChatOpenAI)', '#e74c3c', 'Generates\nraw text response'),
    ('Output\nParser', '#2ecc71', 'Extracts\nstructured data'),
    ('Output\n(Python obj)', '#f39c12', 'String, JSON,\nor Pydantic model'),
]

n = len(components)
for i, (name, color, desc) in enumerate(components):
    x = 0.08 + i * 0.21
    rect = mpatches.FancyBboxPatch((x, 0.35), 0.16, 0.45,
                                    boxstyle='round,pad=0.02',
                                    facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + 0.08, 0.62, name, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(x + 0.08, 0.25, desc, ha='center', va='center', fontsize=8, color='gray')

    if i < n - 1:
        ax.annotate('', xy=(x + 0.24, 0.58), xytext=(x + 0.16, 0.58),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
        ax.text(x + 0.20, 0.65, '|', ha='center', fontsize=18, fontweight='bold')

ax.text(0.5, 0.92, 'LCEL Chain: prompt | llm | output_parser',
        ha='center', fontsize=13, fontweight='bold',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f8f8f8', edgecolor='gray'))

plt.tight_layout()
plt.savefig('/tmp/langchain_lcel.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Simplest LCEL Chain ───────────────────────────────────────────────
print("=== Simplest LangChain Chain ===")
print()

if CAN_CALL:
    # The 3-step chain
    prompt = ChatPromptTemplate.from_template(
        "Explain {topic} to a 10-year-old in exactly 2 sentences."
    )
    chain = prompt | llm | StrOutputParser()

    # Invoke the chain
    result = chain.invoke({"topic": "black holes"})
    print(f"Topic: black holes")
    print(f"Result: {result}")

    # Batch invocation (multiple inputs at once)
    topics = ["black holes", "photosynthesis", "cryptocurrency"]
    results = chain.batch([{"topic": t} for t in topics])
    print()
    for topic, res in zip(topics, results):
        print(f"[{topic}]: {res[:80]}...")
else:
    print("Code:")
    print("  prompt = ChatPromptTemplate.from_template(")
    print("      'Explain {topic} to a 10-year-old in 2 sentences.'")
    print("  )")
    print("  chain = prompt | llm | StrOutputParser()")
    print("  result = chain.invoke({'topic': 'black holes'})")
    print()
    print("Topic: black holes")
    print("Result: A black hole is a region in space where gravity is so strong that")
    print("        nothing, not even light, can escape from it. They form when massive")
    print("        stars run out of fuel and collapse under their own weight.")
    print()
    print("# chain.batch() runs multiple inputs efficiently (parallel where possible)")
    print("results = chain.batch([{'topic': 'black holes'}, {'topic': 'photosynthesis'}])")

## 3. Prompt Templates — Reusable, Parameterized Prompts

In [ ]:
# ── Prompt Templates: Different Types ─────────────────────────────────

if LC_AVAILABLE:
    # 1. Simple string template
    simple_prompt = PromptTemplate.from_template(
        "Generate a {style} explanation of {topic} in {length} words."
    )
    formatted = simple_prompt.format(style="humorous", topic="DNS", length="50")
    print("1. PromptTemplate (string):")
    print(f"   {formatted}")

    # 2. Chat prompt (for chat models like GPT-3.5/4)
    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a {role} who explains things with {style}."),
        ("human", "Explain: {topic}"),
    ])
    messages = chat_prompt.format_messages(
        role="pirate", style="nautical metaphors", topic="cloud computing"
    )
    print("\n2. ChatPromptTemplate:")
    for msg in messages:
        print(f"   [{msg.__class__.__name__}]: {msg.content}")

    # 3. Few-shot prompt (provide examples for in-context learning)
    few_shot_template = ChatPromptTemplate.from_messages([
        ("system", "Classify the sentiment. Reply with exactly: positive, negative, or neutral."),
        ("human", "This movie was incredible!"),
        ("ai", "positive"),
        ("human", "Worst service I've ever experienced."),
        ("ai", "negative"),
        ("human", "The food was alright."),
        ("ai", "neutral"),
        ("human", "{new_review}"),  # ← The actual query
    ])
    print("\n3. Few-Shot ChatPromptTemplate:")
    print("   System: Classify sentiment...")
    print("   (Human: 'This movie was incredible!' → AI: 'positive')  ← examples")
    print("   (Human: 'Worst service ever' → AI: 'negative')")
    print("   Human: {new_review}  ← actual query")
    print("   (AI generates the classification)")

    if CAN_CALL:
        chain = few_shot_template | llm | StrOutputParser()
        review = "I waited 2 hours and the staff was rude."
        result = chain.invoke({"new_review": review})
        print(f"\n   Test: '{review}' → '{result}'")

else:
    print("Prompt Template Types:")
    print()
    print("1. PromptTemplate.from_template('Explain {topic} in {n} words.')")
    print("   → Simple string with {variable} placeholders")
    print()
    print("2. ChatPromptTemplate.from_messages([('system', ...), ('human', ...)])")
    print("   → Multi-role conversation template for chat models")
    print()
    print("3. Few-shot template: include examples of input→output pairs")
    print("   → Model learns the pattern from examples (no fine-tuning needed!)")
    print()
    print("Why use templates?")
    print("  1. Reusable — define once, use with different inputs")
    print("  2. Composable — templates can include other templates")
    print("  3. Testable — easy to unit test with different inputs")
    print("  4. Type-safe — validates that all required variables are provided")

## 4. Output Parsers — Structured Data from LLM Output

In [ ]:
# ── Output Parsers ────────────────────────────────────────────────────

if LC_AVAILABLE:
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    try:
        from langchain.output_parsers import CommaSeparatedListOutputParser, PydanticOutputParser
        from pydantic import BaseModel, Field
        PARSERS_AVAILABLE = True
    except ImportError:
        PARSERS_AVAILABLE = False

print("=" * 60)
print(" Output Parsers: Getting Structured Data from LLMs")
print("=" * 60)
print()

# Simulated LLM responses
simulated_product_json = """{
  "name": "TechPro Laptop",
  "price": 1299.99,
  "rating": 4.5,
  "pros": ["Fast processor", "Long battery life", "Lightweight"],
  "cons": ["Expensive", "Limited ports"]
}"""

simulated_tags = "machine learning, neural networks, deep learning, AI, Python, data science"

# Parser examples (conceptual — run if LangChain is installed)
examples = [
    {
        "name": "StrOutputParser (default)",
        "desc": "Returns raw string from LLM",
        "code": "chain = prompt | llm | StrOutputParser()",
        "output": "'Hello, here is my answer...'",
        "use_when": "Conversational responses, summaries"
    },
    {
        "name": "JsonOutputParser",
        "desc": "Parses LLM output as JSON dict/list",
        "code": "chain = prompt | llm | JsonOutputParser()",
        "output": "{name: ..., price: 1299.99, pros: [...]}",
        "use_when": "Product info extraction, structured data"
    },
    {
        "name": "CommaSeparatedListOutputParser",
        "desc": "Splits comma-separated output into list",
        "code": "chain = prompt | llm | CommaSeparatedListOutputParser()",
        "output": "['machine learning', 'neural networks', 'deep learning']",
        "use_when": "Tag generation, keyword extraction"
    },
    {
        "name": "PydanticOutputParser",
        "desc": "Parses into a typed Pydantic model (best for complex schemas)",
        "code": "parser = PydanticOutputParser(pydantic_object=Product)",
        "output": "Product(name='TechPro', price=1299.99, rating=4.5)",
        "use_when": "Complex schemas, type validation, data pipelines"
    },
]

for ex in examples:
    print(f"{'─'*60}")
    print(f"Parser: {ex['name']}")
    print(f"  What: {ex['desc']}")
    print(f"  Code: {ex['code']}")
    print(f"  Output: {ex['output']}")
    print(f"  Use: {ex['use_when']}")

print()
print("Best practice: use Pydantic models for reliable structured output")
print()
print('''
from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser

class Product(BaseModel):
    name: str = Field(description="Product name")
    price: float = Field(description="Price in USD")
    pros: list[str] = Field(description="List of advantages")

parser = PydanticOutputParser(pydantic_object=Product)

# The parser adds formatting instructions to the prompt automatically!
prompt = PromptTemplate(
    template="Extract product info from: {text}\\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser
product = chain.invoke({"text": review_text})
print(product.price)  # 1299.99 — strongly typed!
''')

## 5. Memory Systems — Making Conversations Stateful

LLMs are stateless — each API call is independent. Memory systems make them appear stateful by storing and retrieving conversation history.

| Memory Type | Stores | Use Case | Downside |
|-------------|--------|----------|----------|
| `ConversationBufferMemory` | Full history | Short conversations | Grows without limit |
| `ConversationBufferWindowMemory` | Last K messages | Medium conversations | Loses old context |
| `ConversationSummaryMemory` | LLM summary of history | Long conversations | Extra LLM call for summary |
| `ConversationKGMemory` | Knowledge graph of facts | Complex multi-fact convos | Complex setup |
| External (Redis/DB) | Stored in database | Multi-user, persistent | Requires infrastructure |

In [ ]:
# ── Memory Systems ────────────────────────────────────────────────────

# Implement a simple memory system from scratch to understand the concept

class ConversationMemory:
    """
    A simple sliding window conversation memory.

    This is what LangChain's ConversationBufferWindowMemory does under the hood.
    """
    def __init__(self, max_messages=10):
        self.history = []
        self.max_messages = max_messages
        self.total_messages = 0

    def add(self, role, content):
        self.history.append({"role": role, "content": content})
        self.total_messages += 1
        # Keep only last N messages (sliding window)
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages:]

    def get_messages(self, system_prompt=None):
        """Return messages formatted for the API."""
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.extend(self.history)
        return messages

    def summary_stats(self):
        return f"In memory: {len(self.history)}/{self.max_messages} msgs | Total ever: {self.total_messages}"


# LangChain-style memory with manual management
print("=== Memory System Demo ===")
print()

memory = ConversationMemory(max_messages=6)  # Only keep last 6 messages

conversation = [
    ("user", "My name is Alex."),
    ("assistant", "Nice to meet you, Alex! How can I help you today?"),
    ("user", "I'm learning Python. Can you recommend a project?"),
    ("assistant", "Great choice! I recommend building a weather app using an API."),
    ("user", "What API should I use?"),
    ("assistant", "Try OpenWeatherMap — it's free and well-documented."),
    ("user", "Cool! By the way, do you remember my name?"),
    ("assistant", "Of course! You're Alex."),
    ("user", "And what project did you suggest?"),  # Message 9
    ("assistant", "I suggested a weather app."),    # Message 10
]

for role, content in conversation:
    memory.add(role, content)

print(f"After {len(conversation)} exchanges:")
print(f"  {memory.summary_stats()}")
print()
print("Current memory window (what the model sees):")
for msg in memory.get_messages():
    prefix = 'User:' if msg['role'] == 'user' else 'AI:  '
    print(f"  [{msg['role']:<10}] {msg['content']}")

print()
print("Note: The first 4 exchanges are GONE from memory (window size = 6)")
print("If the model is asked 'What's my name?', it may not know — it was in the forgotten part!")
print()
print("Solution: ConversationSummaryMemory — periodically summarize old history with LLM:")
print("  Old: [msg1, msg2, msg3, msg4, ...old messages...]")
print("  → LLM summarizes: 'User named Alex learning Python, using weather app project'")
print("  New: [summary, msg8, msg9, msg10]")
print("This fits much more history in the context window!")

## 6. RAG — Retrieval Augmented Generation

RAG is the most important LangChain pattern. It solves two critical problems:
1. **Knowledge cutoff**: LLMs don't know about events after their training date
2. **Hallucination**: LLMs sometimes confidently state false facts

**RAG Pipeline:**
```
Documents → Split into chunks → Embed chunks → Store in vector DB
                                                        ↓
User query → Embed query → Similarity search in vector DB → Retrieve top-k chunks
                                                        ↓
                           Prompt: "Based on context: [chunks]\nAnswer: {query}"
                                                        ↓
                           LLM generates grounded answer
```

Think of RAG like open-book exam vs closed-book:
- Without RAG: Closed-book — model must rely on memorized knowledge (can forget/hallucinate)
- With RAG: Open-book — model has relevant pages from the textbook in front of it

In [ ]:
# ── RAG from Scratch (without external vector store) ──────────────────
#
# We implement RAG manually to show exactly what LangChain does.
# In production: use FAISS, Chroma, Pinecone, Weaviate, etc.

import re

def simple_embed(text):
    """Hash-based pseudo-embeddings for demonstration."""
    np.random.seed(hash(text) % (2**31))
    vec = np.random.randn(128)
    return vec / np.linalg.norm(vec)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


class SimpleRAG:
    """
    A minimal RAG system built from scratch.
    Shows exactly what LangChain's RetrievalQA chain does internally.

    In production use:
      - Real embeddings (OpenAIEmbeddings, HuggingFace)
      - Real vector stores (FAISS, Chroma, Pinecone)
      - LangChain's built-in splitters and chains
    """

    def __init__(self, chunk_size=200):
        self.chunks = []
        self.chunk_embeddings = []
        self.chunk_size = chunk_size

    def add_document(self, text, source='unknown'):
        """Split document into chunks and embed each chunk."""
        # Simple sentence-based chunking
        sentences = re.split(r'(?<=[.!?]) +', text)
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) < self.chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    self.chunks.append({"text": current_chunk.strip(), "source": source})
                    self.chunk_embeddings.append(simple_embed(current_chunk.strip()))
                current_chunk = sentence

        if current_chunk.strip():
            self.chunks.append({"text": current_chunk.strip(), "source": source})
            self.chunk_embeddings.append(simple_embed(current_chunk.strip()))

    def retrieve(self, query, top_k=3):
        """Find the most relevant chunks for a query."""
        query_emb = simple_embed(query)
        scores = [(cosine_sim(query_emb, emb), i)
                  for i, emb in enumerate(self.chunk_embeddings)]
        scores.sort(reverse=True)
        return [(score, self.chunks[i]) for score, i in scores[:top_k]]

    def query(self, question):
        """RAG: retrieve relevant chunks, then generate answer."""
        # Step 1: Retrieve
        relevant = self.retrieve(question, top_k=2)
        context = "\n\n".join([
            f"[Source: {r['source']}]\n{r['text']}" for _, r in relevant
        ])

        # Step 2: Build RAG prompt
        rag_prompt = f"""Answer the question based ONLY on the provided context.
If the context doesn't contain enough information, say "I don't have enough context to answer this."

Context:
{context}

Question: {question}

Answer:"""

        # Step 3: Generate (simulate here)
        return rag_prompt, relevant


# Build a RAG system for a fictional company's knowledge base
rag = SimpleRAG(chunk_size=200)

documents = [
    ("""TechCorp was founded in 2015 by Sarah Chen and James Liu. The company specializes in
     enterprise software solutions. TechCorp has offices in San Francisco, London, and Singapore.
     The company employs over 500 people worldwide and serves 2000+ enterprise clients.""",
     "company_overview.pdf"),

    ("""TechCorp's flagship product is DataFlow Pro, a data pipeline automation tool.
     DataFlow Pro supports 50+ data connectors and processes over 1 billion records daily.
     Key features include real-time monitoring, automatic scaling, and GDPR compliance tools.
     DataFlow Pro starts at $299/month for teams.""",
     "product_spec.pdf"),

    ("""Our return policy allows customers to return products within 30 days for a full refund.
     Enterprise contracts have a 90-day satisfaction guarantee. All refunds are processed
     within 5-7 business days. To initiate a return, contact support@techcorp.com.""",
     "support_policy.pdf"),
]

for text, source in documents:
    rag.add_document(text, source)

print(f"Knowledge base: {len(rag.chunks)} chunks from {len(documents)} documents")
print()

# Test queries
test_questions = [
    "Who founded TechCorp?",
    "What is the price of DataFlow Pro?",
    "How long do I have to return a product?",
    "What is the weather in San Francisco?",  # Not in the knowledge base!
]

print("=== RAG Retrieval Results ===")
for q in test_questions:
    prompt, relevant = rag.query(q)
    print(f"\nQuestion: {q}")
    print(f"Top retrieved chunks:")
    for score, chunk in relevant:
        print(f"  [{score:.3f}] [{chunk['source']}] {chunk['text'][:80]}...")

print()
print("RAG prevents hallucination: the model can only answer from retrieved context.")
print("For 'weather in SF' — no relevant context is found, so the model says it doesn't know.")

In [ ]:
# ── LangChain's Built-in RAG (production code) ────────────────────────

print("=== LangChain Production RAG Pipeline ===")
print()

if CAN_CALL:
    try:
        from langchain_community.document_loaders import TextLoader
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        from langchain_community.vectorstores import FAISS
        from langchain.chains import RetrievalQA

        # Create a test document
        test_doc = """
TechCorp DataFlow Pro is an enterprise data pipeline tool.
It was founded by Sarah Chen in 2015 and processes 1 billion records daily.
The product costs $299/month for teams.
Returns are accepted within 30 days.
Support email is support@techcorp.com.
        """

        with open('/tmp/techcorp_kb.txt', 'w') as f:
            f.write(test_doc)

        # 1. Load documents
        loader = TextLoader('/tmp/techcorp_kb.txt')
        documents = loader.load()

        # 2. Split into chunks
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
        chunks = splitter.split_documents(documents)

        # 3. Create vector store
        vectorstore = FAISS.from_documents(chunks, embeddings_model)

        # 4. Create RAG chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type='stuff',  # 'stuff' = jam all context into prompt
            retriever=vectorstore.as_retriever(search_kwargs={'k': 2})
        )

        # Test
        result = qa_chain.invoke({'query': 'Who founded TechCorp?'})
        print(f"Question: Who founded TechCorp?")
        print(f"Answer: {result['result']}")

    except Exception as e:
        print(f"Error: {e}")
        print("Install missing deps: pip install faiss-cpu langchain-community")

else:
    code = '''
# The complete 4-step LangChain RAG pipeline:

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import RetrievalQA

# Step 1: Load documents (PDF, TXT, Web, etc.)
loader = PyPDFLoader("your_document.pdf")
documents = loader.load()  # List of Document objects

# Step 2: Split into chunks (crucial for long documents)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Characters per chunk
    chunk_overlap=200   # Overlap to avoid cutting mid-sentence
)
chunks = splitter.split_documents(documents)

# Step 3: Embed and store in vector database
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("my_knowledge_base")  # Persist to disk

# Step 4: Create RAG chain
llm = ChatOpenAI(model="gpt-3.5-turbo")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Concatenate all retrieved chunks
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True  # Know where the answer came from
)

# Query
result = qa_chain.invoke({"query": "What is the refund policy?"})
print(result["result"])              # The answer
print(result["source_documents"])    # Which chunks were used
    '''
    print(code)
    print()
    print("Example output:")
    print("  Answer: 'According to the policy document, customers can return products")
    print("           within 30 days for a full refund.'")
    print("  Sources: [Document(page_content='return policy...', metadata={'source': 'policy.pdf'})]")

## 7. LangChain Agents — LLMs That Plan and Act

An **agent** is an LLM that decides WHAT to do, step by step, based on available tools and the current situation. Unlike a chain (which follows a fixed sequence), an agent **reasons** about what to do next.

The most popular agent architecture is **ReAct** (Reason + Act):
```
Thought: I need to find the weather in Paris.
Action: get_weather(location="Paris")
Observation: {temp: 18°C, condition: cloudy}
Thought: Now I have the weather. I should also check the forecast.
Action: get_forecast(location="Paris", days=3)
Observation: [{day: Mon, temp: 20°C}, ...]
Thought: I have enough information to answer.
Final Answer: The weather in Paris is currently 18°C and cloudy...
```

The LLM generates both the Thought and the Action. You execute the action and provide the Observation. This loop continues until the LLM decides it has enough info for a Final Answer.

In [ ]:
# ── LangChain Agent (ReAct pattern) ───────────────────────────────────

print("=== LangChain ReAct Agent Demo ===")
print()

if CAN_CALL:
    try:
        from langchain.agents import create_react_agent, AgentExecutor
        from langchain.tools import tool
        from langchain import hub

        @tool
        def get_word_length(word: str) -> int:
            """Returns the number of characters in a word."""
            return len(word)

        @tool
        def multiply_numbers(a: float, b: float) -> float:
            """Multiplies two numbers together."""
            return a * b

        tools = [get_word_length, multiply_numbers]

        # Get the ReAct prompt template from LangChain Hub
        prompt = hub.pull("hwchase17/react")

        agent = create_react_agent(llm, tools, prompt)
        agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

        result = agent_executor.invoke({
            "input": "How many letters are in 'LangChain'? Multiply that by 3."
        })
        print(f"\nFinal answer: {result['output']}")

    except Exception as e:
        print(f"Agent error: {e}")

else:
    # Simulate the ReAct trace
    print("Input: 'How many letters are in LangChain? Multiply that by 3.'")
    print()
    trace = [
        ("Thought", "I need to find the number of letters in 'LangChain'."),
        ("Action", "get_word_length('LangChain')"),
        ("Observation", "9"),
        ("Thought", "The word 'LangChain' has 9 letters. Now I need to multiply 9 by 3."),
        ("Action", "multiply_numbers(9, 3)"),
        ("Observation", "27"),
        ("Thought", "I now have all the information needed to answer."),
        ("Final Answer", "'LangChain' has 9 letters. 9 multiplied by 3 equals 27."),
    ]

    for step_type, content in trace:
        prefix = '→' if step_type in ['Action', 'Final Answer'] else ' '
        color_indicator = '  [Tool call]  ' if step_type == 'Action' else ''
        print(f"  {prefix} [{step_type}]{color_indicator}: {content}")

print()
print("Key insight: The LLM controls the LOOP — it decides when to stop.")
print("Chains: Fixed sequence of steps")
print("Agents: Dynamic — LLM decides what to do next at each step")
print()
print("Real-world agent tools could be:")
print("  - Search the web (Tavily, Google Search API)")
print("  - Query a database (SQL tool)")
print("  - Send an email")
print("  - Run Python code")
print("  - Browse URLs")
print("  - Call any REST API")

## 8. Common Pitfalls

In [ ]:
print("=" * 68)
print(" LangChain Common Pitfalls")
print("=" * 68)

pitfalls = [
    {
        "title": "1. Breaking API changes (LangChain updates frequently)",
        "fix": "Pin versions: pip install langchain==0.x.x langchain-openai==0.x.x",
        "why": "LangChain moved from monolithic to modular (langchain-core, langchain-openai, etc.). Old tutorials break."
    },
    {
        "title": "2. Using deprecated chains (LLMChain, ConversationalRetrievalChain)",
        "fix": "Use LCEL: chain = prompt | llm | parser instead of LLMChain",
        "why": "LangChain deprecated many high-level abstractions in favor of LCEL (more composable)."
    },
    {
        "title": "3. Not chunking documents — stuffing entire docs into prompt",
        "fix": "Always use RecursiveCharacterTextSplitter before embedding",
        "why": "A 50-page PDF doesn't fit in GPT's context window. You MUST split."
    },
    {
        "title": "4. Chunk size too large — retrieval finds irrelevant content",
        "fix": "Start with chunk_size=500, chunk_overlap=50 and tune",
        "why": "Large chunks have mixed content. Small, focused chunks improve retrieval precision."
    },
    {
        "title": "5. Agent loops forever (max_iterations not set)",
        "fix": "AgentExecutor(max_iterations=10, max_execution_time=30)",
        "why": "Agents can get stuck in loops. Always set a hard limit on iterations AND time."
    },
    {
        "title": "6. Not using LangSmith for debugging",
        "fix": "Set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY in env",
        "why": "Complex chains are hard to debug. LangSmith shows every step, input, output, and latency."
    },
]

for p in pitfalls:
    print(f"\n{'─'*68}")
    print(f"  {p['title']}")
    print(f"  Fix:  {p['fix']}")
    print(f"  Why:  {p['why']}")

print(f"\n{'='*68}")

## 9. Mini Project: Complete Document Q&A System

In [ ]:
# ── Mini Project: Document Q&A with Source Attribution ───────────────

class DocumentQASystem:
    """
    A complete RAG-based Q&A system that:
    1. Ingests multiple documents
    2. Answers questions with context
    3. Attributes answers to source documents
    4. Indicates when it doesn't know

    Uses our SimpleRAG for demonstration (replace with LangChain+FAISS in production).
    """

    def __init__(self):
        self.rag = SimpleRAG(chunk_size=300)
        self.conversation_history = []
        self.query_count = 0

    def load_documents(self, docs):
        """Load (text, source) pairs into the knowledge base."""
        for text, source in docs:
            self.rag.add_document(text, source)
        print(f"Loaded {len(docs)} documents → {len(self.rag.chunks)} chunks")

    def answer(self, question, threshold=0.2):
        """Answer a question using RAG."""
        self.query_count += 1

        # Retrieve relevant chunks
        relevant = self.rag.retrieve(question, top_k=3)

        # Check confidence
        best_score = relevant[0][0] if relevant else 0

        if best_score < threshold:
            return {
                "answer": "I don't have enough information in my knowledge base to answer this question confidently.",
                "sources": [],
                "confidence": "low",
                "best_score": best_score
            }

        # Build context
        context_parts = []
        sources = set()
        for score, chunk in relevant[:2]:
            if score > threshold:
                context_parts.append(f"[{chunk['source']}]: {chunk['text']}")
                sources.add(chunk['source'])

        context = "\n\n".join(context_parts)

        # Generate answer (simulated for this demo)
        if CAN_CALL:
            messages = [{
                "role": "system",
                "content": "Answer based on the provided context. Be concise and cite the source."
            }, {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }]
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model='gpt-3.5-turbo', messages=messages, max_tokens=200
            )
            answer_text = resp.choices[0].message.content
        else:
            # Simulate finding the relevant info in context
            answer_text = f"Based on the provided documentation: {context_parts[0][:150]}..."

        return {
            "answer": answer_text,
            "sources": list(sources),
            "confidence": "high" if best_score > 0.5 else "medium",
            "best_score": best_score
        }


# Build the system
qa = DocumentQASystem()

# Load company knowledge base
company_docs = [
    ("TechCorp was founded in 2015. Our headquarters is in San Francisco. "
     "We have 500+ employees and offices in London and Singapore. "
     "CEO is Sarah Chen, CTO is James Liu.", "about_us.txt"),

    ("DataFlow Pro is our flagship product. It starts at $299/month for teams. "
     "Enterprise pricing is custom. 14-day free trial available. "
     "Supports 50+ data connectors including Salesforce, Snowflake, BigQuery.", "product.txt"),

    ("Support hours: Monday-Friday 9am-6pm Pacific Time. "
     "Emergency support (P0 issues) available 24/7 for Enterprise customers. "
     "Contact: support@techcorp.com or call 1-800-TECHCORP.", "support.txt"),

    ("Return policy: 30-day money-back guarantee for all plans. "
     "Refunds processed within 5-7 business days. "
     "Annual subscriptions refunded pro-rated.", "refunds.txt"),
]

qa.load_documents(company_docs)

# Test with various questions
test_questions = [
    "Who is the CEO of TechCorp?",
    "How much does DataFlow Pro cost?",
    "What are your support hours?",
    "Do you offer cryptocurrency payments?",  # Not in docs
    "Can I get a refund after 60 days?",
]

print("=" * 65)
print(" TechCorp Document Q&A System")
print("=" * 65)
print()

for question in test_questions:
    result = qa.answer(question)
    print(f"Q: {question}")
    print(f"A: {result['answer'][:150]}")
    print(f"   [Confidence: {result['confidence']} | Score: {result['best_score']:.3f} | Sources: {result['sources']}]")
    print()

print("System stats:")
print(f"  Total queries: {qa.query_count}")
print(f"  Knowledge chunks: {len(qa.rag.chunks)}")
print(f"  Documents loaded: {len(company_docs)}")

## 10. Interview Q&A

---

### Q1: What is LCEL and why was it introduced?
**A**: LCEL (LangChain Expression Language) is a declarative way to compose LangChain components using the `|` pipe operator. It was introduced to replace the older imperative chain construction (LLMChain, etc.) with something more composable, type-safe, and streaming-friendly. Benefits: (1) All LCEL chains support streaming by default, (2) Components are type-checked at composition time, (3) Batch execution and async are built-in, (4) Easy to add middleware (logging, retry) anywhere in the chain.

---

### Q2: What is the difference between a Chain and an Agent?
**A**: A **chain** is a fixed, predetermined sequence of steps: A → B → C. You define the flow at code time, and it always runs the same way. An **agent** uses the LLM to dynamically decide what to do next: it has a set of tools and chooses which to call based on the current situation. Chains: predictable, fast, good for structured tasks. Agents: flexible, slower (more LLM calls), good for open-ended tasks. Use chains when you know the exact steps; use agents when the steps depend on the data.

---

### Q3: How does RAG prevent hallucination?
**A**: RAG (Retrieval Augmented Generation) injects retrieved documents into the prompt so the model has factual context to reference. The key is the prompt engineering: "Answer ONLY based on the provided context. If the context doesn't contain the answer, say you don't know." The model is no longer relying on memorized (possibly wrong) knowledge — it's reading the relevant passages right there in the prompt. Additionally, you can ask the model to cite its sources, making hallucinations detectable when the cited text doesn't actually say what the model claims.

---

### Q4: What is chunk size and overlap, and how do you choose them?
**A**: When splitting documents for RAG, **chunk size** determines how much text per chunk (in characters or tokens). **Overlap** ensures consecutive chunks share some text so that sentences split at a boundary appear in both chunks. Guidelines: Chunk size 500-1000 chars works for most use cases. Larger chunks = more context per retrieval but lower precision. Smaller chunks = more precise retrieval but may miss necessary context. Overlap of 10-20% of chunk size prevents losing information at boundaries. Tune based on your documents and query types.

---

### Q5: LangChain vs LlamaIndex — when do you choose each?
**A**: LlamaIndex is optimized for **data indexing and retrieval** — it has more sophisticated indexing strategies (tree index, list index, knowledge graph), better handling of complex documents, and richer query engines. LangChain is optimized for **LLM application orchestration** — chains, agents, tools, memory, and multi-step workflows. In practice: Use LlamaIndex if your primary challenge is retrieving information from complex documents. Use LangChain if you're building multi-step workflows, agents, or need rich tool integrations. They're also complementary — many production systems use both.

---

### Q6: What is the ReAct agent pattern?
**A**: ReAct (Reason + Act) is an agent architecture where the LLM alternates between reasoning (Thought: what should I do?) and acting (Action: call this tool with these args), observing the result (Observation: the tool returned X), and repeating until it has enough information for a Final Answer. The key insight is that reasoning traces and actions are interleaved — the model can plan, act, observe, update its plan, and act again. This enables multi-step problem solving impossible with single-shot prompts.

## 11. Resources

### Official
- **LangChain Docs**: https://python.langchain.com/
- **LCEL Guide**: https://python.langchain.com/docs/expression_language/
- **LangSmith (debugging)**: https://smith.langchain.com/
- **GitHub**: https://github.com/langchain-ai/langchain

### Tutorials
- **LangChain Crash Course**: https://www.youtube.com/watch?v=aywZrzNaKjs
- **RAG from Scratch**: https://www.youtube.com/watch?v=sVcwVQRHIc8
- **LangChain Agents**: https://www.youtube.com/watch?v=DWUdGFCeGVQ

### Papers
- **ReAct (Reason+Act)**: https://arxiv.org/abs/2210.03629
- **RAG paper**: https://arxiv.org/abs/2005.11401
- **Self-RAG**: https://arxiv.org/abs/2310.11511

---

## Summary

| Concept | Takeaway |
|---------|----------|
| LCEL | `prompt | llm | parser` — composable pipeline with `|` |
| PromptTemplate | Reusable prompts with `{variable}` placeholders |
| OutputParser | Convert LLM text → Python types (str, JSON, Pydantic) |
| Memory | Store conversation history to maintain context |
| RAG | Retrieve docs → inject as context → LLM answers from facts |
| Agents | LLM decides dynamically what tools to call |
| Chunking | Split docs into ~500 char chunks before embedding |

**Next**: LlamaIndex — advanced data indexing strategies for production RAG systems!